# 面试问题：IVF-PQ 向量检索如何训练、编码和查询？

可以直接复述的回答是：IVF 先用 coarse k-means 把向量分到倒排簇，查询只探测最近的 `nprobe` 个簇。PQ 再把 residual 切成多个子空间，每段用小码本索引替代浮点向量。查询时为每个被探测簇建立 ADC 距离查找表，文档距离等于各段查表值之和。`nlist`、`nprobe`、子空间数和码本大小共同控制延迟、内存与召回。最常见失败是边界查询的真近邻落在第二近 coarse 簇，`nprobe=1` 会在 PQ 打分前就把它漏掉。下面只用 NumPy 手写 k-means、residual 编码和查表距离。

## 真实案例：十二件商品的离线语义向量召回

向量是教学构造的四维离线 embedding，四维分别近似音频、影像、计算和便携语义。商品名称真实可读，但数值不是任何线上模型输出，仅用于解释 ANN 机制。

In [1]:
import numpy as np  # 导入 NumPy 实现向量距离与码本训练
np.set_printoptions(precision=3, suppress=True)  # 固定中间矩阵显示精度
products = [  # 定义十二件商品及其四维教学向量
    ("V-01", "入耳降噪耳机", [0.2, 0.1, 0.0, 0.8]),  # 便携音频商品
    ("V-02", "头戴监听耳机", [0.4, 0.0, 0.1, 0.3]),  # 专业音频商品
    ("V-03", "户外蓝牙音箱", [0.7, 0.2, 0.0, 0.6]),  # 户外音频商品
    ("V-04", "运动相机", [4.8, 5.2, 0.2, 0.9]),  # 便携影像商品
    ("V-05", "微单相机", [5.3, 5.0, 0.4, 0.3]),  # 高画质影像商品
    ("V-06", "手机稳定器", [4.4, 4.6, 0.3, 0.8]),  # 影像配件商品
    ("V-07", "轻薄笔记本", [9.7, 0.4, 5.0, 0.8]),  # 便携计算商品
    ("V-08", "游戏笔记本", [10.4, 0.3, 5.6, 0.2]),  # 高性能计算商品
    ("V-09", "迷你主机", [9.5, 0.2, 4.7, 0.4]),  # 桌面计算商品
    ("V-10", "旅行充电宝", [1.3, 0.4, 0.2, 5.1]),  # 旅行电源商品
    ("V-11", "磁吸移动电源", [1.0, 0.3, 0.1, 5.5]),  # 便携电源商品
    ("V-12", "户外电源", [1.8, 0.6, 0.4, 5.8]),  # 户外大容量电源
]  # 结束十二件商品
vectors = np.asarray([row[2] for row in products], dtype=np.float64)  # 构造十二乘四向量矩阵
product_ids = [row[0] for row in products]  # 保存向量行号到商品 ID 映射
product_names = {row[0]: row[1] for row in products}  # 建立商品可读名称索引
manual_queries = [  # 定义五个可解释查询向量
    ("耳机通勤", np.array([0.25, 0.10, 0.05, 0.75])),  # 接近入耳耳机的查询
    ("拍摄骑行", np.array([4.75, 5.10, 0.20, 0.95])),  # 接近运动相机的查询
    ("高性能游戏电脑", np.array([10.30, 0.35, 5.50, 0.25])),  # 接近游戏笔记本的查询
    ("磁吸旅行电源", np.array([1.05, 0.30, 0.10, 5.40])),  # 接近磁吸移动电源的查询
    ("专业监听", np.array([0.42, 0.02, 0.10, 0.28])),  # 接近头戴监听耳机的查询
]  # 结束五个常规查询
print("输入预览：id | name | embedding")  # 输出商品向量表头
for product_id, name, vector in products:  # 逐条展示十二件商品
    print(f"{product_id} | {name:8} | {vector}")  # 展示业务名称和完整四维向量
print("向量矩阵形状：", vectors.shape)  # 展示索引训练输入规模

输入预览：id | name | embedding
V-01 | 入耳降噪耳机   | [0.2, 0.1, 0.0, 0.8]
V-02 | 头戴监听耳机   | [0.4, 0.0, 0.1, 0.3]
V-03 | 户外蓝牙音箱   | [0.7, 0.2, 0.0, 0.6]
V-04 | 运动相机     | [4.8, 5.2, 0.2, 0.9]
V-05 | 微单相机     | [5.3, 5.0, 0.4, 0.3]
V-06 | 手机稳定器    | [4.4, 4.6, 0.3, 0.8]
V-07 | 轻薄笔记本    | [9.7, 0.4, 5.0, 0.8]
V-08 | 游戏笔记本    | [10.4, 0.3, 5.6, 0.2]
V-09 | 迷你主机     | [9.5, 0.2, 4.7, 0.4]
V-10 | 旅行充电宝    | [1.3, 0.4, 0.2, 5.1]
V-11 | 磁吸移动电源   | [1.0, 0.3, 0.1, 5.5]
V-12 | 户外电源     | [1.8, 0.6, 0.4, 5.8]
向量矩阵形状： (12, 4)


## Baseline / 基线：Exact Search 扫描全部十二个向量

精确基线计算查询到每个商品的平方欧氏距离，作为 ANN 召回的权威近邻和扫描成本对照。

In [2]:
def exact_search(query, top_k=3):  # 实现全量平方欧氏距离搜索
    distances = ((vectors - query) ** 2).sum(axis=1)  # 一次计算查询到十二个向量的精确距离
    order = np.argsort(distances)[:top_k]  # 选择距离最小的前 k 个行号
    return [(product_ids[index], float(distances[index])) for index in order], len(vectors)  # 返回可读商品和扫描数量
print("query | exact top3 | scanned")  # 输出精确搜索结果表头
for query_name, query_vector in manual_queries:  # 遍历五个可解释查询
    exact_results, scanned = exact_search(query_vector)  # 执行十二向量全扫描
    print(f"{query_name:8} | {exact_results} | {scanned}")  # 展示权威近邻和固定扫描成本

query | exact top3 | scanned
耳机通勤     | [('V-01', 0.007500000000000003), ('V-03', 0.2375), ('V-02', 0.23750000000000002)] | 12
拍摄骑行     | [('V-04', 0.015000000000000081), ('V-06', 0.4049999999999997), ('V-05', 0.7749999999999997)] | 12
高性能游戏电脑  | [('V-08', 0.024999999999999856), ('V-07', 0.9150000000000018), ('V-09', 1.3250000000000006)] | 12
磁吸旅行电源   | [('V-11', 0.012499999999999933), ('V-10', 0.17250000000000043), ('V-12', 0.9024999999999995)] | 12
专业监听     | [('V-02', 0.001199999999999997), ('V-03', 0.22319999999999995), ('V-01', 0.33520000000000005)] | 12


## 核心实现：coarse k-means、residual PQ 与 ADC

coarse 码本使用 4 个簇，PQ 将四维 residual 分成两个二维子空间，每段训练 4 个中心，因此每个商品只需两个小整数码。

In [3]:
def squared_distances(points, centers):  # 计算点集到中心集的两两平方距离
    return ((points[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)  # 利用广播形成距离矩阵
def kmeans(points, cluster_count, initial_indices, iterations=20):  # 手写确定性 k-means 训练
    centers = points[np.asarray(initial_indices)].copy()  # 从指定样本初始化中心保证可复现
    assignment = np.zeros(len(points), dtype=np.int64)  # 初始化每个样本的簇编号
    for iteration in range(iterations):  # 执行固定次数 Lloyd 迭代
        assignment = squared_distances(points, centers).argmin(axis=1)  # 将每个点分到最近中心
        new_centers = centers.copy()  # 创建本轮更新后的中心副本
        for cluster_id in range(cluster_count):  # 逐个重算簇均值
            members = points[assignment == cluster_id]  # 取出当前簇成员
            if len(members) > 0:  # 只更新非空簇避免 NaN
                new_centers[cluster_id] = members.mean(axis=0)  # 用成员均值更新中心
        if np.allclose(new_centers, centers):  # 检查中心是否已经收敛
            centers = new_centers  # 保存最终中心
            break  # 提前结束无变化迭代
        centers = new_centers  # 进入下一轮 Lloyd 更新
    return centers, assignment  # 返回码本中心与样本分配
coarse_centers, coarse_assignment = kmeans(vectors, 4, [0, 3, 6, 9])  # 训练四个商品语义粗簇
residuals = vectors - coarse_centers[coarse_assignment]  # 计算每个商品相对 coarse 中心的残差
subspace_slices = [slice(0, 2), slice(2, 4)]  # 把四维 residual 切成两个二维子空间
pq_codebooks = []  # 收集两个子空间的四中心码本
pq_codes = np.zeros((len(vectors), 2), dtype=np.int64)  # 初始化每个商品的两个 PQ 整数码
for subspace_id, subspace in enumerate(subspace_slices):  # 逐段训练 product quantizer
    codebook, codes = kmeans(residuals[:, subspace], 4, [0, 3, 6, 9])  # 在当前二维残差上训练四中心码本
    pq_codebooks.append(codebook)  # 保存当前子空间码本
    pq_codes[:, subspace_id] = codes  # 保存十二件商品的当前段编码
ivf_lists = {cluster_id: np.where(coarse_assignment == cluster_id)[0].tolist() for cluster_id in range(4)}  # 建立 coarse 簇到商品行号倒排表
print("coarse centers：")  # 输出粗量化中心标题
for cluster_id, center in enumerate(coarse_centers):  # 遍历四个 coarse 中心
    print(f"list={cluster_id} center={center.tolist()} docs={[product_ids[index] for index in ivf_lists[cluster_id]]}")  # 展示每个倒排簇成员
print("PQ codebooks 与商品码：")  # 输出子空间码本标题
for subspace_id, codebook in enumerate(pq_codebooks):  # 遍历两个 PQ 子空间
    print(f"subspace={subspace_id} codebook={codebook.tolist()}")  # 展示四个二维残差中心
for index in range(len(products)):  # 遍历十二件商品编码
    print(f"{product_ids[index]} -> list={coarse_assignment[index]} code={pq_codes[index].tolist()}")  # 展示 IVF 与 PQ 紧凑表示

coarse centers：
list=0 center=[0.43333333333333335, 0.10000000000000002, 0.03333333333333333, 0.5666666666666668] docs=['V-01', 'V-02', 'V-03']
list=1 center=[4.833333333333333, 4.933333333333333, 0.30000000000000004, 0.6666666666666666] docs=['V-04', 'V-05', 'V-06']
list=2 center=[9.866666666666667, 0.3, 5.1000000000000005, 0.4666666666666666] docs=['V-07', 'V-08', 'V-09']
list=3 center=[1.3666666666666665, 0.4333333333333333, 0.23333333333333336, 5.466666666666666] docs=['V-10', 'V-11', 'V-12']
PQ codebooks 与商品码：
subspace=0 codebook=[[-0.3888888888888888, -0.18888888888888877], [0.42500000000000004, 0.0833333333333335], [-0.1444444444444448, 0.12222222222222251], [-0.04999999999999988, -0.06666666666666665]]
subspace=1 codebook=[[0.024999999999999984, 0.18333333333333346], [-0.26666666666666683, -0.016666666666666247], [-0.10000000000000028, 0.28333333333333344], [0.1583333333333331, -0.31666666666666654]]
V-01 -> list=0 code=[2, 0]
V-02 -> list=0 code=[3, 3]
V-03 -> list=0 code=[1, 

## ADC 查询与候选中间量

对每个 probed list，查询先减该 list 的 coarse center，再分别计算两个子空间到四个 PQ 中心的距离表。候选商品只需按两个 code 查表相加。

In [4]:
def ivfpq_search(query, nprobe=1, top_k=3, verbose=False):  # 手写 IVF-PQ 非对称距离查询
    coarse_distance = ((coarse_centers - query) ** 2).sum(axis=1)  # 计算查询到四个粗中心的距离
    probed_lists = np.argsort(coarse_distance)[:nprobe].tolist()  # 选择最近的 nprobe 个倒排簇
    approximate = []  # 收集候选商品的 PQ 近似距离
    lookup_tables = {}  # 保存每个 probed list 的子空间距离表
    for cluster_id in probed_lists:  # 逐个处理被探测的 coarse 簇
        query_residual = query - coarse_centers[cluster_id]  # 计算查询相对当前中心的 residual
        tables = []  # 收集两个子空间的四项查找表
        for subspace_id, subspace in enumerate(subspace_slices):  # 遍历两个 residual 子空间
            table = ((pq_codebooks[subspace_id] - query_residual[subspace]) ** 2).sum(axis=1)  # 计算查询段到四个码字的距离
            tables.append(table)  # 保存当前子空间 ADC 查找表
        lookup_tables[cluster_id] = tables  # 记录当前 coarse 簇完整查找表
        for vector_index in ivf_lists[cluster_id]:  # 只遍历当前倒排簇候选
            distance = sum(tables[subspace_id][pq_codes[vector_index, subspace_id]] for subspace_id in range(2))  # 按两个整数码查表求和
            approximate.append((product_ids[vector_index], float(distance), cluster_id, pq_codes[vector_index].tolist()))  # 保存商品、距离、簇和编码
    approximate.sort(key=lambda item: (item[1], item[0]))  # 按 PQ 近似距离升序排序
    if verbose:  # 根据教学开关输出完整中间过程
        print("coarse distances：", coarse_distance.tolist(), "probed：", probed_lists)  # 展示粗筛选过程
        for cluster_id in probed_lists:  # 遍历每个探测簇
            print(f"list={cluster_id} ADC tables={[table.tolist() for table in lookup_tables[cluster_id]]}")  # 展示查询查找表
        print("candidate | adc_distance | list | codes")  # 输出候选打分表头
        for row in approximate:  # 遍历全部探测候选
            print(row)  # 展示距离如何由整数码产生
    return approximate[:top_k], probed_lists, len(approximate)  # 返回前 k 名、探测簇和候选数
sample_result, sample_lists, sample_count = ivfpq_search(manual_queries[0][1], 1, 3, True)  # 对耳机查询展示完整 ADC 轨迹
print("耳机查询 IVF-PQ top3：", sample_result, "候选数：", sample_count)  # 展示近似召回结果

coarse distances： [0.06749999999999998, 44.43749999999999, 118.10305555555558, 23.638611111111103] probed： [0]
list=0 ADC tables=[[0.077932098765432, 0.377013888888889, 0.016450617283950666, 0.022222222222222258], [6.944444444444412e-05, 0.12027777777777765, 0.023611111111111215, 0.27006944444444414]]
candidate | adc_distance | list | codes
('V-01', 0.01652006172839511, 0, [2, 0])
('V-02', 0.2922916666666664, 0, [3, 3])
('V-03', 0.37708333333333344, 0, [1, 0])
耳机查询 IVF-PQ top3： [('V-01', 0.01652006172839511, 0, [2, 0]), ('V-02', 0.2922916666666664, 0, [3, 3]), ('V-03', 0.37708333333333344, 0, [1, 0])] 候选数： 3


## 失败案例与修正、六查询结果表

我们在固定网格中选择一个 coarse 边界查询：其精确近邻属于第二近 coarse list。`nprobe=1` 在 PQ 计算前就漏掉真近邻；扩大到 2 后恢复候选。网格选择是确定性的，并打印实际向量与簇距离。

In [5]:
boundary_query = None  # 初始化待寻找的边界查询
boundary_exact_id = None  # 初始化边界查询精确近邻身份
for first in np.linspace(0.0, 10.0, 41):  # 在第一维商品语义范围建立固定网格
    for second in np.linspace(0.0, 5.0, 21):  # 在第二维商品语义范围建立固定网格
        candidate_query = np.array([first, second, 2.5, 2.5])  # 构造跨品类模糊查询向量
        exact_row, scanned = exact_search(candidate_query, 1)  # 获取当前网格点真近邻
        exact_index = product_ids.index(exact_row[0][0])  # 定位真近邻向量行号
        closest_list = int(((coarse_centers - candidate_query) ** 2).sum(axis=1).argmin())  # 获取 nprobe 一时选择的粗簇
        if coarse_assignment[exact_index] != closest_list:  # 检查真近邻是否落在第二或更远 coarse 簇
            boundary_query = candidate_query  # 保存首个确定性边界反例
            boundary_exact_id = exact_row[0][0]  # 保存反例真近邻身份
            break  # 结束当前第二维网格搜索
    if boundary_query is not None:  # 检查是否已经找到反例
        break  # 结束第一维网格搜索
all_queries = manual_queries + [("跨品类边界", boundary_query)]  # 形成六个最终评估查询
rows = []  # 收集精确搜索和两种 nprobe 结果
print("query | exact_top1 | nprobe1_top1 | nprobe2_top1 | candidates1/2")  # 输出六查询对照表头
for query_name, query_vector in all_queries:  # 遍历五个常规查询和一个边界反例
    exact_results, scanned = exact_search(query_vector, 1)  # 获取全扫描真近邻
    result_one, lists_one, count_one = ivfpq_search(query_vector, 1, 3)  # 执行低成本单簇搜索
    result_two, lists_two, count_two = ivfpq_search(query_vector, 2, 3)  # 执行扩大探测范围搜索
    top_one = result_one[0][0] if result_one else None  # 提取 nprobe 一近似第一名
    top_two = result_two[0][0] if result_two else None  # 提取 nprobe 二近似第一名
    candidate_ids_one = {row[0] for row in ivfpq_search(query_vector, 1, len(vectors))[0]}  # 获取单簇全部候选身份
    candidate_ids_two = {row[0] for row in ivfpq_search(query_vector, 2, len(vectors))[0]}  # 获取双簇全部候选身份
    rows.append((query_name, exact_results[0][0], top_one, top_two, exact_results[0][0] in candidate_ids_one, exact_results[0][0] in candidate_ids_two, count_one, count_two))  # 保存候选召回与排名
    print(f"{query_name:8} | {exact_results[0][0]} | {top_one} | {top_two} | {count_one}/{count_two}")  # 展示成本与近似结果
boundary_row = rows[-1]  # 读取边界失败样本结果
boundary_coarse_distances = ((coarse_centers - boundary_query) ** 2).sum(axis=1)  # 计算反例到四个 coarse 中心距离
candidate_recall_one = sum(row[4] for row in rows) / len(rows)  # 计算 nprobe 一的真近邻候选召回
candidate_recall_two = sum(row[5] for row in rows) / len(rows)  # 计算 nprobe 二的真近邻候选召回
print("边界向量：", boundary_query.tolist(), "精确近邻：", boundary_exact_id, "coarse距离：", boundary_coarse_distances.tolist())  # 展示失败为何发生
print(f"候选 recall@1：nprobe=1 {candidate_recall_one:.1%}，nprobe=2 {candidate_recall_two:.1%}")  # 汇总扩大探测范围的修正效果

query | exact_top1 | nprobe1_top1 | nprobe2_top1 | candidates1/2
耳机通勤     | V-01 | V-01 | V-01 | 3/6
拍摄骑行     | V-04 | V-04 | V-04 | 3/6
高性能游戏电脑  | V-08 | V-08 | V-08 | 3/6
磁吸旅行电源   | V-11 | V-11 | V-11 | 3/6
专业监听     | V-02 | V-02 | V-02 | 3/6
跨品类边界    | V-06 | V-01 | V-06 | 3/6
边界向量： [0.0, 4.5, 2.5, 2.5] 精确近邻： V-06 coarse距离： [29.37, 31.749999999999996, 125.88555555555557, 32.34444444444444]
候选 recall@1：nprobe=1 83.3%，nprobe=2 100.0%


## 结果解读

IVF 将十二个向量缩小到少量候选，PQ 只读取两个整数码并查四项小表。边界失败发生在 PQ 排序之前，所以换更精细 PQ 码本也救不回被 coarse 阶段丢弃的真近邻；必须调大 nprobe 或改善 coarse 训练。

## 生产边界

教学实验只有四维、四个 coarse 簇和两字节风格编码，没有 SIMD、OPQ、GPU kernel、分片或删除更新。生产索引需要独立训练集、向量归一化、码本版本、增量段、重建策略和 exact holdout recall 曲线，并同时测候选数、P95 延迟和内存。动态数据分布漂移会让 coarse list 失衡。

## 最小回归测试

In [6]:
assert len(products) >= 6 and len(all_queries) >= 6  # 保证案例含多个商品与查询
assert pq_codes.shape == (len(products), 2)  # 保证每个商品实际编码为两个 PQ 子码
assert all(0 <= int(code) < 4 for code in pq_codes.ravel())  # 保证所有子码指向四中心码本
assert boundary_query is not None and boundary_row[4] is False  # 保证 nprobe 一边界漏召回真实复现
assert boundary_row[5] is True  # 保证扩大到两个 coarse 簇后找回真近邻候选
assert candidate_recall_two > candidate_recall_one  # 保证同一六查询候选召回得到改善
assert max(row[7] for row in rows) < len(vectors)  # 保证 nprobe 二仍未退化为全量扫描